## Donut 🍩, Document understanding transformer

![model image](https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/model_doc/donut_architecture.jpg)

[paper: OCR-free Document Understanding Transformer](https://huggingface.co/papers/2111.15664)

[clovaai/donut Github](https://github.com/clovaai/donut?tab=readme-ov-file)

[donut-base model on huggingface](https://huggingface.co/naver-clova-ix/donut-base/)


### TEXT FORMAT

**during training:**

- **INPUT:**
  - Image embedding
  - task_start_token + bos_token + json2token(ground_truth_json) + eos_token

**during inference:**

- **INPUT:**
  - Image embedding
  - task_start_token

- **OUTPUT:**
  - ground_truth_json

In [ ]:
!pip install transformers==4.56.1 --quiet

In [ ]:
!git clone https://github.com/hftuner/clovaai-donut.git /content/hftuner

In [ ]:
import os
import json
from datasets import load_dataset

dataset = load_dataset('hf-tuner/rvl-cdip-document-classification')

class_names = dataset['train'].features['label'].names
id2label = {i: c for i, c in enumerate(class_names)}

# add ground truth
def add_gt(example):
  class_name = id2label[example['label']]
  example["ground_truth"] = json.dumps({
      "gt_parse":{
          "class": class_name # str
      }
  })
  return example

dataset = dataset.map(add_gt)

In [ ]:
dataset

In [ ]:
dataset_info = dataset['train'].features
dataset_info

In [ ]:
ignore_id = -100
task_start_token = "<classification>"
new_special_tokens = []

# add tokens from class_names
for class_name in class_names:
  class_name = class_name.replace(" ","_")
  new_special_tokens.append(class_name)

new_special_tokens.extend([task_start_token])
new_special_tokens

In [ ]:
from hftuner.donut import DataProcessor

data_processor = DataProcessor()
json2token = data_processor.json2token

example_document_classification_gt = {"class" : "scientific_report"} #gt_parse

json2token(example_document_classification_gt)

## special tokens from dataset

In [ ]:
def clean_docs_for_donut(sample):
    gt = json.loads(sample["ground_truth"])
    parsed_text = gt['gt_parse'] if 'gt_parse' in gt else gt['gt_parses']
    text = json2token(parsed_text, update_special_tokens_for_json_key=True) # adds a new token for each key
    return {"text": text}

columns_to_remove = [ c for c in dataset['train'].column_names if c not in ['image', 'ground_truth', 'text']]

data_processor.clear_new_special_tokens()
proc_dataset = dataset.map(clean_docs_for_donut, remove_columns=columns_to_remove)
new_key_tokens = data_processor.get_new_special_tokens()

# print(f"Sample:\n{proc_dataset['train'][3]['text']}")
print(f"Sample:\n{proc_dataset['train'][4]}")
new_key_tokens
# bos_token and eos_token will be added by data collator function

## load model and processor

In [ ]:
import torch
from transformers import DonutProcessor, VisionEncoderDecoderConfig
from hftuner.donut import DonutModel

# ckpt = 'naver-clova-ix/donut-base'
ckpt = 'hf-tuner/donut-classification-turbo'
processor = DonutProcessor.from_pretrained(ckpt, use_fast=True)
model = DonutModel.from_pretrained(ckpt)

len(processor.tokenizer)

### Configure processor and model for finetuning

In [ ]:
new_special_tokens = list(set(new_special_tokens + new_key_tokens))
print(f"New special tokens:  {new_special_tokens}")

# add new special tokens to tokenizer
print(f"Adding {len(new_special_tokens)} special tokens")
processor.tokenizer.add_special_tokens({"additional_special_tokens": new_special_tokens})
print(f"New tokenizer length: {len(processor.tokenizer)}")


In [ ]:
# to find max_token to output during training
def add_token_count(example):
  input_ids = processor.tokenizer(
      example['text'],
      add_special_tokens=True,
      # return_tensors="pt",
  ).input_ids
  example['tokens_count'] = len(input_ids)
  return example

proc_dataset = proc_dataset.map(add_token_count)
max_length = max(proc_dataset['train']['tokens_count'])
print('maximum length of example:', max_length)


In [ ]:
max_length = 8
processor_image_size = [960, 1280] # (width, height)


## UPDATE: processor
processor.image_processor.size['width'] = processor_image_size[0]
processor.image_processor.size['height'] = processor_image_size[1]
processor.image_processor.do_align_long_axis = False # don't rotate image if height is greater than width

## UPDATE: model

# generation config
model.config.decoder.max_length = max_length
model.config.pad_token_id = processor.tokenizer.pad_token_id
# ⚠️ IMPORTANT: set decoder_start_token
task_token_id = processor.tokenizer.encode(task_start_token, add_special_tokens=False)[0]
model.config.decoder_start_token_id = task_token_id

# Resize embedding layer to match vocabulary size
new_emb = model.decoder.resize_token_embeddings(len(processor.tokenizer))

# Adjust our image size and output sequence lengths
model.config.encoder.image_size = processor_image_size[::-1] # (height, width)

print(f"New token embedding size: {new_emb}")

More about : [decoder_start_token_id](https://github.com/huggingface/transformers/blob/4df2529d79d75f44e70396df5888a32ffa02d61e/src/transformers/models/vision_encoder_decoder/modeling_vision_encoder_decoder.py#L331)

## Data Collator

In [ ]:
from torchvision.transforms import Compose, ColorJitter
jitter = Compose(
    [
         ColorJitter(brightness=0.45, contrast=0.35, saturation=0.25, hue=0.4),
    ]
)

In [ ]:
def prepare_data(examples):
    images = [e['image'].convert("RGB") for e in examples]
    # images = [ jitter(img) for img in images] # data_augmentation
    texts = [e['text'] for e in examples]
    # create tensor from image
    pixel_values = processor(
        images,
        return_tensors="pt"
        ).pixel_values
    # tokenize text
    input_ids = processor.tokenizer(
        texts,
        add_special_tokens=True, # add <s> and </s>
        max_length=max_length,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    ).input_ids
    labels = input_ids.clone()
    # ignore pad tokens when calculating `loss`
    labels[labels == processor.tokenizer.pad_token_id] = ignore_id
    return {"pixel_values": pixel_values, "labels": labels}

# TEST
batch= prepare_data([
    proc_dataset['train'][5],
])
batch.keys(), batch['pixel_values'].shape, batch['labels'].shape

## ⚙️ Efficient fine tuning

In [ ]:
f"number of encoder parameters: {(model.encoder.num_parameters() / 1e6):.2f}M"

In [ ]:
for p in model.encoder.parameters():
  p.requires_grad = False

In [ ]:
# trainer.get_num_trainable_parameters() / 1e6 # freezed decoder

In [ ]:
import torch.nn.functional as F

def resize_bart_abs_pos_emb(weight: torch.Tensor, max_length: int) -> torch.Tensor:
  """
  Resize position embeddings
  Truncate if sequence length of Bart backbone is greater than given max_length,
  else interpolate to max_length
  """
  if weight.shape[0] > max_length:
      weight = weight[:max_length, ...]
  else:
      weight = (
          F.interpolate(
              weight.permute(1, 0).unsqueeze(0),
              size=max_length,
              mode="linear",
              align_corners=True,
          )
          .squeeze(0)
          .permute(1, 0)
      )
  return weight

In [ ]:
model.decoder.model.decoder.embed_positions.weight = torch.nn.Parameter(resize_bart_abs_pos_emb(model.decoder.model.decoder.embed_positions.weight, 10 ))
model.config.decoder.max_position_embeddings = 8

In [ ]:
# model.decoder.model.decoder
model.decoder.model.decoder.embed_positions.weight.shape

In [ ]:
# trainer.get_num_trainable_parameters() / 1e6 # reduced decoder.embed_positions

## ⚙️ Train

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
from huggingface_hub import login

save_model_name = "donut-classification-turbo"
# lr used in pre-training: [1e-5, 1e-4]
# use lr below or comparable to pre_training_lr

# Authenticate with Hugging Face Hub
login()

# Arguments for training
training_args = Seq2SeqTrainingArguments(
    output_dir=save_model_name,
    num_train_epochs= 3,
    # max_steps = 100,
    logging_steps=100,
    learning_rate=1e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size = 8,
    weight_decay=0.01, # prevents memorization
    fp16=True,
    eval_strategy="steps",
    eval_steps=2000,
    save_strategy="epoch",
    save_total_limit=1,
    # group_by_length=True,
    predict_with_generate=True,
    remove_unused_columns=False,
    # push to hub parameters
    report_to="tensorboard",
    push_to_hub=True,
    hub_strategy="end",
    hub_private_repo=False,
    hub_model_id=save_model_name,
)

# Create Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    data_collator=prepare_data,
    train_dataset=proc_dataset["train"],
    eval_dataset=proc_dataset["test"],
)

In [ ]:
trainer.get_num_trainable_parameters() / 1e6

In [ ]:
trainer.get_num_trainable_parameters() / 1e6

In [ ]:
# Start training
trainer.train()

In [ ]:
trainer.save_model(save_model_name)
processor.save_pretrained(save_model_name)

# Save processor and create model card
hf_repo_id = f"hf-tuner/{save_model_name}"
trainer.create_model_card()
trainer.push_to_hub()
processor.push_to_hub(repo_id=hf_repo_id)


## 🎯 Evaluate

In [ ]:
!pip install evaluate --quiet

In [ ]:
%cd /content/hftuner/scripts

!python test.py --pretrained_model_name_or_path "hf-tuner/donut-classification-turbo" \
  --dataset_name_or_path "hf-tuner/rvl-cdip-document-classification" \
  --task_start_token "<classification>" \
  --task_type "classification" \
  --eval_batch_size 16 --save_predictions

## 🚀 Inference

In [ ]:
import re
import torch
from transformers import DonutProcessor, VisionEncoderDecoderConfig
from hftuner.donut import DonutModel

ckpt = "/content/donut-efficient-classifier-test"
# ckpt = "hf-tuner/donut-base-finetuned-rvl-cdip"

config = VisionEncoderDecoderConfig.from_pretrained(ckpt)
processor = DonutProcessor.from_pretrained(ckpt)
device = "cuda" if torch.cuda.is_available() else "cpu"
config.dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model = DonutModel.from_pretrained(ckpt, config=config)
model.to(device)


In [ ]:
from datasets import load_dataset

dataset = load_dataset('hf-tuner/rvl-cdip-document-classification')

In [ ]:
# inference (generation)
task_start_token = "<classification>"
sample = dataset['test'][21]

pixel_values = processor(sample['image'],return_tensors="pt").pixel_values.to(device)

decoder_input_ids = processor.tokenizer(task_start_token, add_special_tokens=False, return_tensors="pt").input_ids.to(device)

generated_ids = model.generate(pixel_values,
                               decoder_input_ids=decoder_input_ids,
                               max_length=16,
                               bad_words_ids=[[processor.tokenizer.unk_token_id]]
                               )

print(f'generated tokens count: {len(generated_ids[0])}')
generated_text = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
print(generated_text)
processor.token2json(generated_text)

In [ ]:
print(sample['label'])
# sample['image'].resize((500,800))